# 04-4. 인코딩·bytes와 바이너리 구조 실습

## Goal

문자열과 바이트의 경계를 구분하고, 인코딩·바이트 순서·고정 길이 구조를 검증하며 학습용 바이너리 패킷을 안전하게 읽습니다. 각 단계는 **결과 예측 → 실행 → 이유 설명 → 입력 변경** 순서로 진행하세요.


## Setup

모든 파일은 운영 파일과 분리된 임시 디렉터리에 만듭니다. 이 노트북은 Python 3.10 이상의 표준 라이브러리만 사용하며 위에서 아래로 순서대로 실행합니다.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import struct

binary_tempdir = TemporaryDirectory(prefix="python-04-4-")
lab_dir = Path(binary_tempdir.name)
assert lab_dir.is_dir()


def expect_exception(exception_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except exception_type as exc:
        return exc
    raise AssertionError(f"{exception_type.__name__}이 발생해야 합니다")


print("격리 실습 디렉터리:", lab_dir)


## Steps

### 1. str과 bytes의 변환 방향

str은 문자 의미를 다루고 bytes는 파일과 프로토콜의 원시 바이트를 다룹니다. 아래 값을 실행하기 전에 자료형과 길이를 먼저 예상하세요.


In [ ]:
sample_text = "Python 한글"
sample_bytes = sample_text.encode("utf-8")
restored_text = sample_bytes.decode("utf-8")

assert isinstance(sample_text, str)
assert isinstance(sample_bytes, bytes)
assert restored_text == sample_text
assert len(sample_bytes) > len(sample_text)

print("문자 수:", len(sample_text))
print("바이트 수:", len(sample_bytes))
print("16진수:", sample_bytes.hex())


### 2. 입력을 바꾸어 문자 수와 바이트 수 비교

ASCII, 한글, 이모지, 결합 문자의 길이를 비교합니다. 목록에 문자를 하나 더 넣고 같은 규칙이 성립하는지 확인해 보세요.


In [ ]:
characters = ["A", "가", "😊", "é"]
byte_lengths = {
    character: len(character.encode("utf-8"))
    for character in characters
}

assert byte_lengths["A"] == 1
assert byte_lengths["가"] == 3
assert byte_lengths["😊"] == 4
assert len("é") == 2
assert len("é".encode("utf-8")) == 3

for character in characters:
    print(repr(character), "문자열 길이=", len(character), "UTF-8 바이트=", byte_lengths[character])


### 3. 텍스트 모드·바이너리 모드·BOM

텍스트는 인코딩을 명시해 읽고, 원본 바이트가 의미를 가지는 데이터는 바이너리 모드로 읽습니다. utf-8-sig는 입력 계약이 BOM을 허용할 때만 선택합니다.


In [ ]:
text_path = lab_dir / "hello.txt"
binary_path = lab_dir / "sample.bin"
bom_path = lab_dir / "with-bom.txt"

text_path.write_text("안녕하세요\n", encoding="utf-8")
binary_path.write_bytes(b"\x4d\x5a\x90\x00")
bom_path.write_text("BOM 허용 입력", encoding="utf-8-sig")

loaded_text = text_path.read_text(encoding="utf-8")
loaded_bytes = binary_path.read_bytes()
loaded_bom_text = bom_path.read_text(encoding="utf-8-sig")

assert loaded_text == "안녕하세요\n"
assert loaded_bytes == b"MZ\x90\x00"
assert bom_path.read_bytes().startswith(b"\xef\xbb\xbf")
assert loaded_bom_text == "BOM 허용 입력"

print(loaded_text.rstrip())
print(loaded_bytes.hex(" "))


### 4. bytes 조회와 손실 있는 디코딩 정책

bytes 인덱싱은 정수, 슬라이싱은 새 bytes를 반환합니다. errors="replace" 결과는 화면 미리보기에는 쓸 수 있지만 원본을 대신할 수 없습니다.


In [ ]:
header = b"MZ\x90\x00"
mutable_header = bytearray(b"ABC")
mutable_header[0] = ord("Z")

invalid_utf8 = b"\xff\xfe"
decode_error = expect_exception(
    UnicodeDecodeError,
    invalid_utf8.decode,
    "utf-8",
)
preview = invalid_utf8.decode("utf-8", errors="replace")

assert header[0] == 77
assert header[:2] == b"MZ"
assert bytes(mutable_header) == b"ZBC"
assert decode_error.start == 0
assert preview == "��"
assert invalid_utf8 == b"\xff\xfe"

print("헤더:", header.hex(" "))
print("손실 미리보기:", preview)


### 5. tell·seek·read와 짧은 읽기

read(size)는 최대 size바이트를 반환합니다. 고정 길이 필드는 요청한 길이와 실제 길이가 같은지 검사합니다.


In [ ]:
def read_exact(file, size):
    if size < 0:
        raise ValueError("읽기 크기는 0 이상이어야 합니다")

    data = file.read(size)
    if len(data) != size:
        raise ValueError(
            f"필드 길이 부족: 필요={size}, 실제={len(data)}"
        )
    return data


with binary_path.open("rb") as file:
    assert file.tell() == 0
    first_four = read_exact(file, 4)
    assert file.tell() == 4
    file.seek(0)
    first_two = read_exact(file, 2)

with binary_path.open("rb") as file:
    short_read_error = expect_exception(ValueError, read_exact, file, 5)

negative_size_error = expect_exception(ValueError, read_exact, None, -1)

assert first_four == b"MZ\x90\x00"
assert first_two == b"MZ"
assert "필드 길이 부족" in str(short_read_error)
assert "0 이상" in str(negative_size_error)

print(first_four.hex(" "), first_two)


### 6. 바이트 순서와 struct

같은 네 바이트도 형식 명세의 바이트 순서에 따라 값이 달라집니다. 네이티브 정렬 대신 리틀 엔디언 또는 빅 엔디언을 명시합니다.


In [ ]:
raw_number = b"\x3c\x00\x00\x00"
little_value = int.from_bytes(raw_number, byteorder="little", signed=False)
big_value = int.from_bytes(raw_number, byteorder="big", signed=False)

HEADER = struct.Struct("<2sI")
packed_header = HEADER.pack(b"LB", little_value)
magic, payload_length = HEADER.unpack(packed_header)

assert little_value == 60
assert big_value == 1_006_632_960
assert HEADER.size == 6
assert magic == b"LB"
assert payload_length == 60
assert little_value.to_bytes(4, byteorder="little") == raw_number

print("little:", little_value, "big:", big_value)
print("헤더:", packed_header.hex(" "))


### 7. 학습용 바이너리 패킷

패킷은 식별자 2바이트, 리틀 엔디언 4바이트 길이, UTF-8 페이로드로 구성합니다. 식별자·크기 상한·선언 길이·실제 파일 크기를 확인한 뒤에만 페이로드를 읽습니다.


In [ ]:
MAX_PAYLOAD_SIZE = 1024


def write_learning_packet(path, payload, *, magic=b"LB"):
    if not isinstance(payload, bytes):
        raise TypeError("payload는 bytes여야 합니다")
    path.write_bytes(HEADER.pack(magic, len(payload)) + payload)


def read_learning_packet(path):
    with path.open("rb") as file:
        file.seek(0, 2)
        file_size = file.tell()
        file.seek(0)

        header_bytes = read_exact(file, HEADER.size)
        magic, payload_length = HEADER.unpack(header_bytes)

        if magic != b"LB":
            raise ValueError("알 수 없는 파일 식별자입니다")
        if payload_length > MAX_PAYLOAD_SIZE:
            raise ValueError("허용된 페이로드 크기를 초과했습니다")
        if HEADER.size + payload_length != file_size:
            raise ValueError("선언된 길이와 실제 파일 크기가 다릅니다")

        payload = read_exact(file, payload_length)

    return payload


### 8. 정상 입력을 변경해 다시 검증

payload_text 값을 다른 UTF-8 문자열로 바꾸고 길이 필드와 복원 결과가 함께 달라지는지 확인하세요.


In [ ]:
payload_text = "안전한 바이트"
payload = payload_text.encode("utf-8")
packet_path = lab_dir / "packet.bin"

write_learning_packet(packet_path, payload)
loaded_payload = read_learning_packet(packet_path)

assert loaded_payload == payload
assert loaded_payload.decode("utf-8") == payload_text
assert packet_path.stat().st_size == HEADER.size + len(payload)

print("페이로드:", loaded_payload.decode("utf-8"))
print("파일 크기:", packet_path.stat().st_size)


## Checks

### 9. 정상·오류·경계 입력 자기점검

잘못된 식별자, 잘린 헤더, 과도한 길이, 선언 길이 불일치, 잘못된 UTF-8을 각각 독립된 작은 파일로 재현합니다.


In [ ]:
bad_magic_path = lab_dir / "bad-magic.bin"
truncated_path = lab_dir / "truncated.bin"
oversized_path = lab_dir / "oversized.bin"
wrong_length_path = lab_dir / "wrong-length.bin"
invalid_text_path = lab_dir / "invalid-text.bin"

write_learning_packet(bad_magic_path, b"ok", magic=b"XX")
truncated_path.write_bytes(b"LB")
oversized_path.write_bytes(
    HEADER.pack(b"LB", MAX_PAYLOAD_SIZE + 1)
)
wrong_length_path.write_bytes(
    HEADER.pack(b"LB", 5) + b"ab"
)
write_learning_packet(invalid_text_path, b"\xff")

bad_magic_error = expect_exception(
    ValueError, read_learning_packet, bad_magic_path
)
truncated_error = expect_exception(
    ValueError, read_learning_packet, truncated_path
)
oversized_error = expect_exception(
    ValueError, read_learning_packet, oversized_path
)
wrong_length_error = expect_exception(
    ValueError, read_learning_packet, wrong_length_path
)

original_invalid_bytes = invalid_text_path.read_bytes()
invalid_payload = read_learning_packet(invalid_text_path)
invalid_decode_error = expect_exception(
    UnicodeDecodeError, invalid_payload.decode, "utf-8"
)

assert "식별자" in str(bad_magic_error)
assert "필드 길이 부족" in str(truncated_error)
assert "초과" in str(oversized_error)
assert "실제 파일 크기" in str(wrong_length_error)
assert invalid_decode_error.start == 0
assert invalid_text_path.read_bytes() == original_invalid_bytes

print("오류 행렬 5종을 통과했습니다.")


In [ ]:
final_checks = {
    "str과 bytes 구분": isinstance(sample_text, str) and isinstance(sample_bytes, bytes),
    "BOM 계약": loaded_bom_text == "BOM 허용 입력",
    "짧은 읽기 거부": "필드 길이 부족" in str(short_read_error),
    "명시적 바이트 순서": little_value == 60 and big_value != little_value,
    "정상 패킷 왕복": loaded_payload == payload,
    "오류 파일 원본 보존": invalid_text_path.read_bytes() == original_invalid_bytes,
}

for name, passed in final_checks.items():
    assert passed
    print(f"[PASS] {name}")


## Next Steps

- 텍스트 의미를 다룰 때와 원본 바이트를 다룰 때의 모드를 구분해 설명합니다.
- 디코딩 실패 시 원본 bytes를 보존하고 손실 정책을 명시합니다.
- 고정 구조는 길이·식별자·바이트 순서·크기 상한을 먼저 검증합니다.
- 다음 절에서는 이 인코딩·줄바꿈 계약을 CSV 읽기와 쓰기에 적용합니다.


In [ ]:
binary_tempdir.cleanup()
assert not lab_dir.exists()
print("임시 실습 디렉터리를 정리했습니다.")
